# 🧪 scratch_inspect — your playground

Not a guided tour (that's `skeleton_explore.ipynb`). This is a **blank bench**: imports
are pre-wired, the rest is yours to poke. Nothing here is committed as evidence — break
things freely, re-run cells, delete and rewrite.

**The three learning surfaces, and the order to use them:**

| # | Surface | Use it to… | How |
|---|---|---|---|
| 1 | **`skeleton_explore.ipynb`** (the *explain* notebook) | see how the pieces click together, once | open it, Run All, read the markdown between cells |
| 2 | **this notebook** (the *scratch* notebook) | poke any Python module live, on your own questions | edit cells below, re-run |
| 3 | **the terminal** (`forge inspect` / `cast`) | inspect + drive the real Solidity contract | see the last cell |

Do them in that order when a topic is new: **watch it (1) → poke it (2) → touch the real thing (3)**.
Once a topic is familiar, jump straight to 2 or 3.

In [ ]:
# --- the microscope is pre-focused: these are the same imports the guided tours use ---
from a2a_interfaces import EntitlementReader, NetworkProvisioner  # the two ports (Protocols)
from a2a_interfaces import fixtures as fx                          # the canonical example, one source of truth
from a2a_interfaces import models                                 # every shape as a pydantic class

from e2e.skeleton.fakes import FakeChain, FakeClock, FakeNet, OfferAlreadyUsed
from e2e.skeleton.scripted_agents import ScriptedConsumer, ScriptedProvider
from e2e.skeleton.stub_controller import StubController, predicate
from e2e.skeleton.worlds import MockWorld, ChainWorld              # one test surface, two realities (M1.5)

from chainmcp import ChainClient, ChainRevert, offer_digest        # the real adapter (M1.5)
from chainmcp.testing import ANVIL_KEYS, launch_anvil              # throwaway live chains

# handy: everything the fixtures module exposes (Ada, Bell, the offer, the window, ...)
print([n for n in dir(fx) if not n.startswith("_")])

## A. Poke a shape (the `Offer`)

`.model_dump()` = see the data · `.model_fields` = see the schema · `type(x)` / `dir(x)` = what is this?

In [ ]:
o = fx.CANONICAL_OFFER
o.model_dump()                 # the 12 fields as plain data
# next, try:
# models.Offer.model_fields    # the schema itself, field by field
# o.model_dump(by_alias=True)  # camelCase — the exact bytes Solidity will verify
# o.consumer                   # who may redeem it? (0x000... = anyone)
# o.salt                       # the field that makes it single-use

## B. Poke a fake (the `FakeChain` — "the blockchain is a dict")

Build one, look at its guts, run `fulfill`, watch the dict change.

In [ ]:
clock = FakeClock(fx.WINDOW.start - 60)        # 1 min before the window opens
chain = FakeChain(clock, balances={fx.ADA: fx.PRICE_10_TOK, fx.BELL: 0})

# is it really an EntitlementReader? (satisfied by shape, not inheritance)
print("FakeChain is an EntitlementReader?", isinstance(chain, EntitlementReader))
# poke its state:
# chain.balances
# chain.__dict__          # everything the fake keeps — note how little there is
# vars(chain).keys()

## C. Poke the predicate (the bouncer's checklist, a pure function)

`predicate(...)` takes facts and returns `ok` or an `ErrorCode`. Flip one fact, watch the verdict change.

In [ ]:
# inspect its signature before calling it:
import inspect
print(inspect.signature(predicate))
print(inspect.getsource(predicate))   # read the ~10 lines that decide everything
# then call it with the canonical view and flip one field at a time to see each denial

## D. Poke a live chain (since M1.5 the bench has real hardware too)

Uncomment to stand up a throwaway Anvil + contracts and get Ada's client. Ideas:
`ada.chain_time()` · `ada.faucet(10**19); ada.tok_balance(ada.address)` ·
`ada.offer_digest(fx.CANONICAL_OFFER).hex()` · sign with Bell, fulfill as Ada, `ada.get(1)`.
The guided tour is `chain_client_explore.ipynb`; clean up with `anvil.stop()` when done.

In [ ]:
# anvil = launch_anvil(timestamp=fx.WINDOW.start - 1680)   # 13:32 story time
# ada = ChainClient(anvil.rpc_url, ANVIL_KEYS["ada"], deployment=anvil.deployment)
# ...
# anvil.stop()

## E. Surface 3 — the terminal (Solidity: inspect, don't test)

`forge inspect` / `cast` are to a contract what `.model_dump()` / `dir()` are to a Python object.
Run these in a terminal from `contracts/` (no test suite needed):

```bash
cd contracts
forge inspect A2ASettlement storageLayout   # what lives on-chain (slots 0-5 = OZ's, 6-7 = yours)
forge inspect A2ASettlement abi             # the full interface
forge inspect A2ASettlement methods         # callable functions + selectors
```

For *behavior* on a live chain, follow the scripted labs in order:
**`contracts/EXPLORE-settlement.md`** (mint + read, M1.2) →
**`EXPLORE-fulfill.md`** (sign + redeem + cheats, M1.3) →
**`EXPLORE-revoke.md`** (fine print, expiry, kill switch, M1.4).

## F. Free space

Your own questions go here. Add cells. Break things. This notebook is disposable.